# Semantic Search RAG Application
This notebook performs semantic search over a video transcript and answers questions using the Groq API and strictly in-memory embeddings.

In [3]:
# Install necessary dependencies
!python3 -m pip install sentence-transformers scikit-learn numpy groq

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
import os
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq


In [7]:
class TranscriptEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        """Initialize the sentence transformer model."""
        self.model = SentenceTransformer(model_name)
    
    def clean_text(self, text: str) -> str:
        """Removes extra spaces and line breaks from the text."""
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def chunk_text(self, text: str, sentences_per_chunk: int = 3) -> list:
        """Splits cleaned text into chunks of roughly `sentences_per_chunk` sentences."""
        text = self.clean_text(text)
        
        # Simple sentence splitting on punctuation
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk = " ".join(sentences[i:i+sentences_per_chunk])
            chunks.append(chunk)
            
        return chunks

    def embed_chunks(self, chunks: list) -> list:
        """Creates embeddings for a list of text chunks and stores them in memory."""
        if not chunks:
            return []
            
        embeddings = self.model.encode(chunks)
        
        stored_data = []
        for text, emb in zip(chunks, embeddings):
            stored_data.append({
                "text": text,
                "embedding": emb
            })
            
        return stored_data
    
    def embed_query(self, query: str):
        """Creates an embedding for a single user query."""
        return self.model.encode([query])[0]

In [8]:
def get_top_k(query_embedding: np.ndarray, stored_data: list, top_k: int = 3) -> list:
    """
    Computes cosine similarity between the query embedding and all stored chunk embeddings.
    Returns the top_k most similar chunks.
    """
    if not stored_data:
        return []
        
    # Extract all embeddings into a 2D numpy array
    chunk_embeddings = np.array([item["embedding"] for item in stored_data])
    
    # Reshape query embedding from 1D to 2D for sklearn
    query_vec = query_embedding.reshape(1, -1)
    
    # Compute similarity
    similarities = cosine_similarity(query_vec, chunk_embeddings)[0]
    
    # Get indices of the top-k highest similarity scores
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Format and return the results
    results = []
    for idx in top_indices:
        results.append({
            "text": stored_data[idx]["text"],
            "score": float(similarities[idx])
        })
        
    return results
     

In [9]:
# Example Transcript Data
# You can paste your video transcript here.
transcript = """
Artificial intelligence is intelligence demonstrated by machines, as opposed to intelligence of humans and other animals. Example tasks in which this is done include speech recognition, computer vision, translation between (natural) languages, as well as other mappings of inputs.

The AI applications include advanced web search engines (e.g., Google Search), recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Waymo), generative or creative tools (ChatGPT and AI art), automated decision-making, and competing at the highest level in strategic game systems (such as chess and Go).

As machines become increasingly capable, tasks considered to require \"intelligence\" are often removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is frequently excluded from things considered to be AI, having become a routine technology.
"""

In [10]:
# Initialize the Embedder and Process Transcript
print("Initializing embedding model...")
embedder = TranscriptEmbedder()

print("\nChunking and embedding transcript...")
chunks = embedder.chunk_text(transcript)
stored_data = embedder.embed_chunks(chunks)

print(f"\nTranscript successfully split into {len(chunks)} chunks in-memory.")

Initializing embedding model...

Chunking and embedding transcript...

Transcript successfully split into 2 chunks in-memory.


In [11]:
# Semantic Search & Groq Retrieval Query
query = "What are some examples of AI applications?"

# 1. Retrieve the most relevant contextual chunks
query_emb = embedder.embed_query(query)
top_results = get_top_k(query_emb, stored_data, top_k=1)

print("\n--- Retrieved Context ---")
context_texts = []
for i, res in enumerate(top_results, 1):
    print(f"Chunk {i} (Similarity Score: {res['score']:.4f}):\n{res['text']}\n")
    context_texts.append(res['text'])

combined_context = "\n\n".join(context_texts)

# 2. Formulate prompting logic and submit to Groq
api_key = os.environ.get("GROQ_API_KEY", "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")
if api_key:
    client = Groq(api_key=api_key)
    
    prompt = f"""You are a helpful assistant. Answer the question ONLY using the provided context. If the answer is not in the context, say 'I don't know'.\n\nContext:\n{combined_context}\n\nQuestion:\n{query}\n\nAnswer clearly and concisely."""
    
    print("\n--- Querying Groq LLM ---")
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a precise and helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            model="openai/gpt-oss-120b",
        )
        print("\n--- Final Answer ---")
        print(chat_completion.choices[0].message.content)
    except Exception as e:
        print(f"Error communicating with Groq API: {e}")
else:
    print("\n[!] GROQ_API_KEY environment variable is not set. Skipping the LLM call.")
    print("[!] Set your API key in the environment or directly in Cell 3 to continue.")


--- Retrieved Context ---
Chunk 1 (Similarity Score: 0.6831):
Artificial intelligence is intelligence demonstrated by machines, as opposed to intelligence of humans and other animals. Example tasks in which this is done include speech recognition, computer vision, translation between (natural) languages, as well as other mappings of inputs. The AI applications include advanced web search engines (e.g., Google Search), recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Waymo), generative or creative tools (ChatGPT and AI art), automated decision-making, and competing at the highest level in strategic game systems (such as chess and Go).


--- Querying Groq LLM ---

--- Final Answer ---
Examples of AI applications include:

- Advanced web search engines (e.g., Google Search)  
- Recommendation systems (e.g., YouTube, Amazon, Netflix)  
- Speech‑understanding assistants (e.g., Siri, Alexa)  
- Self‑